# Fabric Client — Iteration Examples

Walk through different patterns for fetching workspace metadata:
serial ``async for`` vs. concurrent ``asyncio.gather`` + ``TaskGroup``.

## Setup & Authentication

Load credentials from environment variables and build the client via
the dependency-injection container.  Log level is set to ``INFO`` to
show operational highlights (scan progress, scoped fetches, timings).

In [ ]:
from collections import defaultdict

import pandas as pd
from dotenv import load_dotenv

from fabric_client import Credentials, FabricContainer, scan

# Load credentials from .env (FABRIC_CLI_TENANT_ID, FABRIC_CLI_CLIENT_ID,
# FABRIC_CLI_CLIENT_SECRET)
load_dotenv()

creds = Credentials.env()

container = FabricContainer()
container.credentials.override(creds)
client = container.client()
client.set_log_level("INFO")

def display(d: defaultdict, name: str) -> pd.DataFrame:
    """Show a collected data category as a DataFrame."""
    return pd.DataFrame(d[name])

## Serial Iteration (``async for``)

Each nested ``async for`` blocks until the inner collection is fully
fetched.  Simple and readable, but every level runs sequentially.

In [ ]:
# Fetch all metadata in nested serial async-for loops

data = defaultdict(list)
filtered = [w for w in await client.workspaces if w.display_name is not None]
async for workspace in scan(filtered):
    data['workspaces'].append(workspace.model_dump())

    async for dataset in workspace.datasets:
        data['datasets'].append(dataset.model_dump())
        async for query in dataset.queries:
            data['queries'].append(query.model_dump())
        async for refresh in dataset.refreshes:
            data['refreshes'].append(refresh.model_dump())

    async for dataflow in workspace.dataflows:
        data["dataflows"].append(dataflow.model_dump())
        async for query in dataflow.queries:
            data["queries"].append(query)
        async for transaction in dataflow.transactions:
            data['transactions'].append(transaction.model_dump())

    async for report in workspace.reports:
        data['reports'].append(report.model_dump())

## Concurrent Iteration (all workspaces at once)

Fan out **every** workspace's datasets, dataflows, reports, refreshes,
queries, and transactions in a single ``TaskGroup``.  One failure never
blocks the rest.  Total time ≈ slowest request across all workspaces.

In [ ]:
# Fully parallel: every workspace × every dataset × every dataflow in
# a single TaskGroup.  Errors are collected so one failure never stops
# the rest.
import asyncio
from typing import Any

concurrent_data = defaultdict(list)
errors: list[dict[str, Any]] = []

def _collect_error(resource_type: str, resource_id: str, exc: BaseException) -> None:
    errors.append({
        "type": resource_type,
        "id": resource_id,
        "error": f"{type(exc).__name__}: {exc}",
    })

async def _gather_dataset_safe(ds):
    try:
        refreshes, queries = await asyncio.gather(
            ds.refreshes, ds.queries,
        )
    except Exception as exc:
        _collect_error("dataset", str(ds.id), exc)
        return ds, [], []
    return ds, refreshes, queries

async def _gather_dataflow_safe(df):
    try:
        queries, transactions = await asyncio.gather(
            df.queries, df.transactions,
        )
    except Exception as exc:
        _collect_error("dataflow", str(df.id), exc)
        return df, [], []
    return df, queries, transactions

async def _gather_workspace(ws):
    """Fetch all scoped resources for a workspace, then fan out child fetches."""
    datasets, dataflows, reports = await asyncio.gather(
        ws.datasets, ws.dataflows, ws.reports,
    )
    ds_coros = [_gather_dataset_safe(ds) for ds in datasets]
    df_coros = [_gather_dataflow_safe(df) for df in dataflows]
    child_results = await asyncio.gather(*ds_coros, *df_coros)
    return ws, datasets, dataflows, reports, child_results, len(datasets)

_filtered = [w for w in await client.workspaces if w.display_name is not None]
workspaces = [ws async for ws in scan(_filtered)]

# Single TaskGroup across ALL workspaces — maximal concurrency
async with asyncio.TaskGroup() as tg:
    tasks = [tg.create_task(_gather_workspace(ws)) for ws in workspaces]

for task in tasks:
    ws, datasets, dataflows, reports, child_results, n_ds = task.result()
    concurrent_data["workspaces"].append(ws.model_dump())

    ds_results = child_results[:n_ds]
    df_results = child_results[n_ds:]

    for ds, refreshes, queries in ds_results:
        concurrent_data["datasets"].append(ds.model_dump())
        for r in refreshes:
            concurrent_data["refreshes"].append(r.model_dump())
        for q in queries:
            concurrent_data["queries"].append(q.model_dump())

    for df, queries, transactions in df_results:
        concurrent_data["dataflows"].append(df.model_dump())
        for q in queries:
            concurrent_data["queries"].append(q.model_dump())
        for t in transactions:
            concurrent_data["transactions"].append(t.model_dump())

    for r in reports:
        concurrent_data["reports"].append(r.model_dump())

print(
    f"datasets={len(concurrent_data['datasets'])}, "
    f"dataflows={len(concurrent_data['dataflows'])}, "
    f"reports={len(concurrent_data['reports'])}, "
    f"refreshes={len(concurrent_data['refreshes'])}, "
    f"transactions={len(concurrent_data['transactions'])}, "
    f"queries={len(concurrent_data['queries'])}"
)

if errors:
    client._logger.warning("%d error(s) collected:", len(errors))
    for err in errors:
        client._logger.warning("  [%s] %s — %s", err["type"], err["id"], err["error"])
else:
    client._logger.info("No errors collected.")

## View Results

Use the ``display()`` helper to inspect collected data as DataFrames.
Change ``concurrent_data`` to ``data`` to view the serial results.

In [ ]:
display(concurrent_data,"workspaces")